# TRIAGE-EG Stage 1 BTC Retrieval Baseline

Uses Stage 0 manifests and BTC keyframes/CLIP only. No dataset-layout rescan, custom keyframes, Fast Line, Event Graph, Agent, VLM, or query reasoning. Text retrieval is protected by the encoder compatibility gate.

Prerequisite: use **Add Input → Notebook Output** to attach the successful saved Stage 0 notebook version. If that version was never saved with outputs, upload the downloaded `triage_eg_stage0_audit_bundle.zip` once as a private Kaggle Dataset. `AIC_STAGE0_BUNDLE` can explicitly select either mounted directory/ZIP. The notebook never reruns Stage 0.

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
REFRESH_REPO = os.environ.get("AIC_REFRESH_REPO", "0") == "1"
DATA_ROOT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
STAGE0_ROOT = Path(os.environ.get("AIC_STAGE0_ROOT", "/kaggle/working/triage_eg_stage0_audit"))
STAGE0_BUNDLE = os.environ.get(
    "AIC_STAGE0_BUNDLE",
    "/kaggle/input/datasets/irthn1311/triage-eg-stage0-audit-bundle",
).strip()
STAGE1_ROOT = Path(
    os.environ.get("AIC_STAGE1_OUTPUT_ROOT", "/kaggle/working/triage_eg_stage1_baseline")
)
BUILD_INDEX = os.environ.get("AIC_STAGE1_BUILD_INDEX", "1") == "1"
REUSE_INDEX = os.environ.get("AIC_STAGE1_REUSE_INDEX", "0") == "1"
if BUILD_INDEX and REUSE_INDEX:
    raise RuntimeError("AIC_STAGE1_BUILD_INDEX and AIC_STAGE1_REUSE_INDEX cannot both be 1")
BACKEND = os.environ.get("AIC_STAGE1_BACKEND", "numpy_exact")
METRIC = os.environ.get("AIC_STAGE1_METRIC", "cosine")
QUERY_TEXT = os.environ.get("AIC_QUERY_TEXT", "").strip()
QUERY_VECTOR = os.environ.get("AIC_QUERY_VECTOR", "").strip()
QUERY_ID = os.environ.get("AIC_QUERY_ID", "notebook_demo")
ENCODER_CONFIG = os.environ.get("AIC_ENCODER_CONFIG", "").strip()
ALLOW_UNVERIFIED = os.environ.get("AIC_ALLOW_UNVERIFIED_ENCODER", "0") == "1"
ZIP_INDEX = os.environ.get("AIC_ZIP_INDEX", "0") == "1"
print(
    {
        "ref": REPO_REF,
        "data": str(DATA_ROOT),
        "stage0": str(STAGE0_ROOT),
        "stage0_bundle": STAGE0_BUNDLE or "AUTO_DISCOVER_KAGGLE_INPUT",
        "stage1": str(STAGE1_ROOT),
        "backend": BACKEND,
        "metric": METRIC,
    }
)

In [ ]:
def git_result(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, check=False)


def git(*args, cwd=None):
    result = git_result(*args, cwd=cwd)
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()


if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout")
if not (REPO_DIR / ".git").is_dir():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    git("clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR))
target = None
if not REFRESH_REPO:
    for candidate in (REPO_REF, f"origin/{REPO_REF}"):
        probe = git_result("rev-parse", "--verify", f"{candidate}^{{commit}}", cwd=REPO_DIR)
        if probe.returncode == 0:
            target = probe.stdout.strip()
            break
if target is None:
    git("fetch", "--depth", "1", "origin", REPO_REF, cwd=REPO_DIR)
    target = "FETCH_HEAD"
git("checkout", "--detach", target, cwd=REPO_DIR)
COMMIT = git("rev-parse", "HEAD", cwd=REPO_DIR)
print("resolved ref:", REPO_REF, "commit:", COMMIT)

In [ ]:
module = REPO_DIR / "src/triage_eg/retrieval/stage1/builder.py"
if not module.is_file():
    raise RuntimeError(f"Missing {module}; publish Stage 1 on {REPO_REF} and refresh")
if not DATA_ROOT.is_dir():
    raise RuntimeError(f"Missing dataset root: {DATA_ROOT}")
sys.path.insert(0, str(REPO_DIR / "src"))
stage0_loader = importlib.import_module("triage_eg.retrieval.stage1.stage0_loader")

STAGE0_ROOT = stage0_loader.resolve_stage0_root(
    STAGE0_ROOT,
    bundle_path=STAGE0_BUNDLE or None,
    search_root=Path("/kaggle/input"),
    excluded_roots=(DATA_ROOT,),
)
print("resolved Stage 0 root:", STAGE0_ROOT)
STAGE1_ROOT.parent.mkdir(parents=True, exist_ok=True)
write_probe = STAGE1_ROOT.parent / ".triage_eg_stage1_write_probe"
try:
    write_probe.write_text("ok", encoding="utf-8")
finally:
    write_probe.unlink(missing_ok=True)
print("NumPy:", np.__version__)
for package in ("faiss", "open_clip", "clip"):
    print(package, "available=", importlib.util.find_spec(package) is not None)
print("No pip install and no model download will be performed.")

In [ ]:
from triage_eg.retrieval.stage1.stage0_loader import load_stage0_bundle

stage0 = load_stage0_bundle(STAGE0_ROOT)
print("BTC gate:", stage0.summary["gates"]["btc_baseline"])
print("rows/videos:", stage0.summary["mapping_rows"], stage0.summary["videos_completed"])
print("duplicate warnings:", stage0.summary["issues"]["by_code"].get("DUPLICATE_FRAME_IDX", 0))
print("original frame:", stage0.contract_notes["original_frame_policy"])
print("CLIP compatibility: UNKNOWN/UNVERIFIED")

In [ ]:
from triage_eg.retrieval.stage1 import Stage1BuildConfig, build_index

if (STAGE1_ROOT / "index/index_manifest.json").is_file() and os.environ.get(
    "AIC_STAGE1_REUSE_INDEX"
) is None:
    REUSE_INDEX = True
    BUILD_INDEX = False
if not BUILD_INDEX and not REUSE_INDEX:
    raise RuntimeError("Enable AIC_STAGE1_BUILD_INDEX or AIC_STAGE1_REUSE_INDEX")
build_config = Stage1BuildConfig(
    stage0_root=STAGE0_ROOT,
    dataset_root=DATA_ROOT,
    output_root=STAGE1_ROOT,
    backend=BACKEND,
    metric=METRIC,
    overwrite=BUILD_INDEX and not REUSE_INDEX,
    reuse_index=REUSE_INDEX,
    strict_root=True,
)
build_result = build_index(build_config)
print("index reused:", build_result.reused)

In [ ]:
manifest = build_result.index_manifest
matrix_path = STAGE1_ROOT / "index/clip_vectors.f16.npy"
print(
    {
        k: manifest[k]
        for k in (
            "vector_count",
            "dimension",
            "dtype",
            "source_clip_files",
            "source_fingerprint",
            "backend",
        )
    }
)
print("index fingerprint:", build_result.summary["index_fingerprint"])
print("matrix size_bytes:", matrix_path.stat().st_size)

In [ ]:
self_report = json.loads((STAGE1_ROOT / "benchmark/self_retrieval_report.json").read_text())
print(self_report)

In [ ]:
encoder_contract = json.loads((STAGE1_ROOT / "encoder/encoder_contract.json").read_text())
compatibility = json.loads((STAGE1_ROOT / "encoder/compatibility_report.json").read_text())
print("encoder contract:", encoder_contract)
print("compatibility:", compatibility)
assert compatibility["compatibility_status"] != "VERIFIED" or encoder_contract[
    "evidence_source"
] in {"AUTHORITATIVE", "EMPIRICAL_PROBE"}

In [ ]:
from triage_eg.retrieval.stage1.contracts import SearchConfig
from triage_eg.retrieval.stage1.encoder import (
    compatibility_gate,
    load_encoder_contract,
    load_text_encoder,
)
from triage_eg.retrieval.stage1.runner import load_query_vector, search_text, search_vector

search_config = SearchConfig(STAGE1_ROOT, QUERY_ID, top_k=100, metric=METRIC)
if QUERY_VECTOR:
    candidates, query_paths = search_vector(load_query_vector(QUERY_VECTOR), search_config)
    query_mode = "VECTOR"
elif QUERY_TEXT:
    contract = load_encoder_contract(ENCODER_CONFIG or None)
    compatibility_gate(contract, allow_unverified=ALLOW_UNVERIFIED)
    encoder = load_text_encoder(contract)
    candidates, query_paths = search_text(
        QUERY_TEXT, search_config, contract, encoder, allow_unverified=ALLOW_UNVERIFIED
    )
    query_mode = "TEXT"
else:
    stored = np.load(STAGE1_ROOT / "index/clip_vectors.f16.npy", mmap_mode="r", allow_pickle=False)
    candidates, query_paths = search_vector(
        np.asarray(stored[0:1], dtype=np.float32), search_config
    )
    query_mode = "STORED_VECTOR_SELF_DEMO"
print("query mode:", query_mode, "(self demo is not semantic text retrieval)")

In [ ]:
for item in candidates[:20]:
    print(
        {
            k: item[k]
            for k in (
                "rank",
                "score",
                "video_id",
                "n",
                "original_frame_idx",
                "pts_time",
                "keyframe_relative_path",
            )
        }
    )

In [ ]:
ranked_videos = [
    json.loads(line)
    for line in query_paths["ranked_videos"].read_text().splitlines()
    if line.strip()
]
for item in ranked_videos[:20]:
    print(item)

In [ ]:
print("candidate exports:")
for name, path in query_paths.items():
    print(name, "->", path)

In [ ]:
from triage_eg.retrieval.stage1.benchmark import run_benchmark

benchmark = run_benchmark(
    STAGE1_ROOT, random_queries=50, self_queries=100, top_k=100, seed=2026, metric=METRIC
)
print(benchmark)

In [ ]:
from zipfile import ZipFile

from triage_eg.retrieval.stage1.writers import create_index_bundle, create_report_bundle

report_zip = Path("/kaggle/working/triage_eg_stage1_baseline_reports.zip")
create_report_bundle(STAGE1_ROOT, report_zip)
with ZipFile(report_zip) as archive:
    report_members = archive.namelist()
assert "index/clip_vectors.f16.npy" not in report_members and report_zip.name not in report_members
print(
    "REPORT ZIP:", report_zip, "size_bytes=", report_zip.stat().st_size, "members=", report_members
)
if ZIP_INDEX:
    estimated = sum(
        (STAGE1_ROOT / name).stat().st_size
        for name in ("index/clip_vectors.f16.npy", "index/vector_norms.f32.npy")
    )
    print("WARNING: creating large index ZIP; uncompressed core bytes=", estimated)
    index_zip = Path("/kaggle/working/triage_eg_stage1_index_bundle.zip")
    create_index_bundle(STAGE1_ROOT, index_zip)
    print("INDEX ZIP:", index_zip, "size_bytes=", index_zip.stat().st_size)
print(
    "This notebook runs the Stage 1 BTC retrieval baseline. It does not create "
    "new keyframes, use a Fast branch, or run Event Graph/Agent logic."
)